# Tratamento da Base de Execução Orçamentária 2024

Notebook para baixar/ler o arquivo `basedadosexecucao_1224.csv`, validar estrutura mínima,
padronizar tipos das colunas de valor (`Vl_*`) e exportar um arquivo final com colunas fixas
para consumo no pipeline do DataSP.

In [1]:
import pandas as pd
from os import environ, makedirs, path

In [2]:
SOURCE_URL = 'https://orcamento.sf.prefeitura.sp.gov.br/orcamento/uploads/2025/basedadosexecucao_1225.csv'


In [3]:

OUTPUT_DIR = path.join('data_output', 'urbanismo')
OUTPUT_FILENAME = 'orcamento_2024.csv'
OUTPUT_PATH = path.join(OUTPUT_DIR, OUTPUT_FILENAME)

In [ ]:
# Definição inicial de colunas (será ajustada conforme arquivo)
EXPECTED_COLUMNS_INITIAL = [
    'Cd_AnoExecucao', 'Cd_Exercicio', 'Cd_Dotacao_Id', 'Administracao', 'Cd_Orgao',
    'Sigla_Orgao', 'Ds_Orgao', 'Cd_Funcao', 'Ds_Funcao', 'Cd_SubFuncao', 'Ds_SubFuncao',
    'Cd_Programa', 'Ds_Programa', 'ProjetoAtividade', 'Ds_Projeto_Atividade',
    'Vl_Orcado_Ano', 'Vl_Suplementado', 'Vl_Reduzido', 'Vl_SuplementadoLiquido',
    'Vl_Orcado_Atualizado', 'Vl_ReservadoLiquido', 'Vl_EmpenhadoLiquido',
    'Vl_Liquidado', 'Vl_Pago'
]

In [ ]:
df_raw = pd.read_csv(SOURCE_URL, sep=';', encoding='latin1', dtype=str)

In [15]:
column_mapping = {
    'COD_EMPRESA_PMSP': 'Cd_AnoExecucao',
    'ANO_EMPENHO': 'Cd_Exercicio',
    'COD_EMPENHO': 'Cd_Dotacao_Id',
    'CÓDIGO_ÓRGÃO': 'Cd_Orgao',
    'SIGLA_ÓRGÃO': 'Sigla_Orgao',
    'DESCRIÇÃO_ÓRGÃO': 'Ds_Orgao',
    'CÓDIGO_FUNÇÃO': 'Cd_Funcao',
    'DESCRIÇÃO_FUNÇÃO': 'Ds_Funcao',
    'CÓDIGO_SUBFUNÇÃO': 'Cd_SubFuncao',
    'DESCRIÇÃO_SUBFUNÇÃO': 'Ds_SubFuncao',
    'CÓDIGO_PROGRAMA': 'Cd_Programa',
    'DESCRIÇÃO_PROGRAMA': 'Ds_Programa',
    'CÓDIGO_PROJ_ATIV': 'ProjetoAtividade',
    'DESCRIÇÃO_PROJ_ATIV': 'Ds_Projeto_Atividade',
    'VALOR_DETALHAMENTO_AÇÃO': 'Vl_Orcado_Ano'
}
EXPECTED_COLUMNS = list(column_mapping.values())
VALUE_COLUMNS = [col for col in EXPECTED_COLUMNS if col.startswith('Vl_')]


In [ ]:
ZERO_TOKENS = {'', '-', '--', '---', 'NA', 'N/A', 'NAN', 'NULL', 'NONE'}

def parse_numeric_column(series: pd.Series) -> tuple[pd.Series, int, int]:
    """Parse coluna com valores monetários para float"""
    original = series.fillna('').astype(str).str.strip()
    normalized = original.str.replace('R$', '', regex=False)
    normalized = normalized.str.replace(' ', '', regex=False)
    normalized = normalized.str.replace('\u00A0', '', regex=False)
    
    has_both = normalized.str.contains('.', regex=False) & normalized.str.contains(',', regex=False)
    normalized.loc[has_both] = normalized.loc[has_both].str.replace('.', '', regex=False)
    normalized = normalized.str.replace(',', '.', regex=False)
    
    numeric = pd.to_numeric(normalized, errors='coerce')
    zero_mask = original.str.upper().isin(ZERO_TOKENS)
    invalid_mask = (~zero_mask) & original.ne('') & numeric.isna()
    
    numeric = numeric.fillna(0.0)
    return numeric, int(invalid_mask.sum()), int(zero_mask.sum())

In [19]:
df_treated = df_raw.rename(columns=column_mapping)

for col in EXPECTED_COLUMNS:
    if col not in df_treated.columns:
        df_treated[col] = pd.NA

df_treated = df_treated[EXPECTED_COLUMNS].copy()

issues = []
for col in VALUE_COLUMNS:
    converted, invalid_count, zero_count = parse_numeric_column(df_treated[col])
    df_treated[col] = converted
    if invalid_count > 0 or zero_count > 0:
        issues.append({
            'coluna': col,
            'inválidos': invalid_count,
            'vazios': zero_count
        })

In [21]:
assert len(df_treated) == len(df_raw), 'Quantidade de linhas alterada!'
assert df_treated.columns.tolist() == EXPECTED_COLUMNS, 'Ordem de colunas inconsistente!'

In [23]:
makedirs(OUTPUT_DIR, exist_ok=True)

df_treated.to_csv(
    OUTPUT_PATH,
    index=False,
    sep=';',
    decimal=',',
    encoding='latin1'
)

In [25]:
df_check = pd.read_csv(OUTPUT_PATH, sep=';', encoding='latin1')

assert df_check.columns.tolist() == EXPECTED_COLUMNS, 'Validação falhou: colunas inconsistentes!'
assert len(df_check) == len(df_treated), 'Validação falhou: quantidade de linhas inconsistente!'

file_size = path.getsize(OUTPUT_PATH) / 1024 / 1024

In [ ]:
pd.DataFrame(issues)


In [ ]:
.